# Эксперимент: признаки появления продажи

Поверх v5. Классификатор "будет ли продажа" - главный рычаг SMAPE (половина скора): ROC AUC около 0.84 против 0.89 у оракула, который знает частоту продаж пары в тесте. В разборе ошибок (`03`) пропущенные продажи часто у пар, которые давно не продавались, - модель не видит событий, после которых продажа вероятнее. Новая группа признаков "появление продажи" (`features.GROUPS`, всё по данным до недели отсчёта, окна <= 28 дней):
- `restock_days` - дней с последнего пополнения (остаток был <= 0 и стал > 0), `restocked7` - было ли оно за неделю;
- `stock_change7` - изменение остатка за неделю (к среднему за 28 дней), `stock_max28` - максимум остатка за 28 дней;
- `art_nz1` - доля складов, где товар продавался на прошлой неделе (раньше был только `art_nz4`).

Варианты: группа у классификатора, у обеих моделей; порог 0.5 и 0.55 (новые признаки могут сдвинуть калибровку вероятностей).

Вариант проходит, если он не хуже v5 (посчитанного в этом же запуске) на обоих сезонных фолдах, а на `recent` хуже не больше чем на 0.005; из прошедших лучший - по среднему двух сезонных. Таблица всех вариантов сохраняется в `grid_<имя>.csv` - потом лучшие идеи собираются вместе.

In [1]:
import sys; sys.path[:0] = ["../src", "../../src"]   # код проекта - в src/ (ноутбук в notebooks/ или notebooks/05_experiments/)

import pandas as pd
import core
from core import *
import classifier, regressor, pipeline

NAME = "05_restock"
# быстрее, чем в 03/04: на Kaggle - каждая 2-я неделя отсчёта (так валидировался v5); деревьев - по 400, как в модулях
if ON_KAGGLE:
    classifier.ORIGIN_STEP = regressor.ORIGIN_STEP = 2
V5 = {"threshold": 0.5, "integer": "smape", "ensemble": [{}, {"random_seed": 7}]}
SEASON = ["season2025", "season2024"]
sales, stock = load_data()
classifier.PARAMS, regressor.PARAMS, classifier.HALF_LIFE, classifier.ORIGIN_STEP

({'iterations': 400,
  'learning_rate': 0.1,
  'depth': 6,
  'loss_function': 'Logloss',
  'max_ctr_complexity': 1,
  'random_seed': 42,
  'verbose': 0,
  'allow_writing_files': False,
  'thread_count': -1,
  'one_hot_max_size': 64,
  'task_type': 'GPU'},
 {'iterations': 400,
  'learning_rate': 0.1,
  'depth': 6,
  'loss_function': 'RMSE',
  'max_ctr_complexity': 1,
  'random_seed': 42,
  'verbose': 0,
  'allow_writing_files': False,
  'thread_count': -1,
  'task_type': 'GPU'},
 26,
 2)

## Варианты

Параметры метода `pipeline`; первый - v5, с ним и сравниваем.

In [2]:
from features import GROUPS
NEW = GROUPS["появление продажи"][0]
CLF = {"features": classifier.FEATURES + NEW}
REG = {"features": regressor.FEATURES + NEW}
VARIANTS = {
    "v5": V5,
    "классификатор + появление продажи": {**V5, "ensemble": [{"clf": CLF}, {"random_seed": 7, "clf": CLF}]},
    "обе модели + появление продажи": {**V5, "ensemble": [{"clf": CLF, "reg": REG}, {"random_seed": 7, "clf": CLF, "reg": REG}]},
}
VARIANTS["классификатор + появление продажи, порог 0.55"] = {**VARIANTS["классификатор + появление продажи"], "threshold": 0.55}
VARIANTS["обе модели + появление продажи, порог 0.55"] = {**VARIANTS["обе модели + появление продажи"], "threshold": 0.55}
list(VARIANTS)

['v5',
 'классификатор + появление продажи',
 'обе модели + появление продажи',
 'классификатор + появление продажи, порог 0.55',
 'обе модели + появление продажи, порог 0.55']

## Сетка на трёх фолдах

Модели учатся один раз на фолд и набор настроек, варианты перебираются поверх кэша. Если вариант не посчитается, он пропускается - остальное считается.

In [3]:
rows = []
for fold, cut in FOLDS.items():
    pairs, s_cut, st_cut, truth, base = core._fold_data(fold, cut)
    for name, params in VARIANTS.items():
        try:
            pred = make_forecast(s_cut, st_cut, "pipeline", pairs=pairs, **params)
        except Exception as e:
            print(f"{fold}, {name}: пропущен - {type(e).__name__}: {e}", flush=True)
            continue
        m = compute_metrics(truth, pred)
        rows.append({"вариант": name, "фолд": fold, **m, "score": score_minmax(m, base),
                     "прогноз / факт": pred.quantity.sum() / truth.quantity.sum()})
    print(fold, "готов", flush=True)
grid = pd.DataFrame(rows)
grid.to_csv(OUT_DIR / f"grid_{NAME}.csv", index=False)
grid.pivot_table(index="вариант", columns="фолд", values=["smape", "wape", "bias", "rmse"]).round(2)

season2025 готов


season2024 готов


recent готов


bias                        \
фолд                                          recent season2024 season2025   
вариант                                                                      
v5                                             76.56      86.47      81.16   
классификатор + появление продажи              76.55      86.54      81.25   
классификатор + появление продажи, порог 0.55  76.48      85.98      81.71   
обе модели + появление продажи                 76.67      86.45      80.78   
обе модели + появление продажи, порог 0.55     76.63      85.92      81.40   

                                                rmse                        \
фолд                                          recent season2024 season2025   
вариант                                                                      
v5                                              3.34       1.93       2.65   
классификатор + появление продажи               3.34       1.93       2.65   
классификатор + появление продажи, порог 0.55   3.34       1.93       2.65   
обе модели + появление продажи                  3.34       1.94       2.69   
обе модели + появление продажи, порог 0.55      3.34       1.95       2.70   

                                               smape                        \
фолд                                          recent season2024 season2025   
вариант                                                                      
v5                                             58.71      42.86      57.59   
классификатор + появление продажи              58.78      42.84      57.52   
классификатор + появление продажи, порог 0.55  58.18      42.69      57.67   
обе модели + появление продажи                 58.76      42.87      57.55   
обе модели + появление продажи, порог 0.55     58.16      42.73      57.68   

                                                wape                        
фолд                                          recent season2024 season2025  
вариант                                                                     
v5                                             47.07      62.92      51.22  
классификатор + появление продажи              47.08      62.86      51.21  
классификатор + появление продажи, порог 0.55  47.01      62.81      51.30  
обе модели + появление продажи                 46.98      63.00      51.35  
обе модели + появление продажи, порог 0.55     46.92      62.96      51.44

## Выбор

Вариант проходит, если он не хуже v5 (посчитанного в этом же запуске) на обоих сезонных фолдах, а на `recent` хуже не больше чем на 0.005; из прошедших лучший - по среднему двух сезонных. Таблица всех вариантов сохраняется в `grid_<имя>.csv` - потом лучшие идеи собираются вместе.

In [4]:
scores = grid.pivot_table(index="вариант", columns="фолд", values="score")[list(FOLDS)]
BASE = scores.loc["v5"]
scores["по сезону"] = scores[SEASON].mean(axis=1)
scores["проходит"] = (scores[SEASON] <= BASE[SEASON]).all(axis=1) & (scores["recent"] <= BASE["recent"] + 0.005)
scores.loc["v5", "проходит"] = False
passed = scores[scores["проходит"]]
BEST = passed["по сезону"].idxmin() if len(passed) else None
print("v5 в этом запуске:", BASE[list(FOLDS)].round(4).to_dict(), "| по сезону", round(BASE[SEASON].mean(), 4))
print("лучший вариант:", BEST, scores.loc[BEST, list(FOLDS) + ["по сезону"]].astype(float).round(4).to_dict() if BEST else "- не прошёл ни один")
scores.sort_values("по сезону").round(4)

v5 в этом запуске: {'season2025': 0.8097, 'season2024': 0.8241, 'recent': 0.7867} | по сезону 0.8169
лучший вариант: классификатор + появление продажи {'season2025': 0.8093, 'season2024': 0.8236, 'recent': 0.7872, 'по сезону': 0.8164}


фолд,season2025,season2024,recent,по сезону,проходит
вариант,,,,,
классификатор + появление продажи,0.8093,0.8236,0.7872,0.8164,True
v5,0.8097,0.8241,0.7867,0.8169,False
"классификатор + появление продажи, порог 0.55",0.8118,0.8222,0.7834,0.8170,False
обе модели + появление продажи,0.8116,0.8253,0.7866,0.8184,False
"обе модели + появление продажи, порог 0.55",0.8141,0.8240,0.7828,0.8191,False
